# Chapter 9.3: Mixed-Precision & Quantization for Recommendation Models

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Apply** FP16/BF16 mixed-precision training to recommendation models with loss scaling
2. **Implement** post-training quantization (int8, int4) for embedding tables
3. **Design** quantization-aware training (QAT) pipelines for rec models
4. **Handle** mixed-precision embedding lookups with FP32 accumulation
5. **Benchmark** speed vs accuracy trade-offs across precision levels
6. **Identify** and resolve gradient overflow issues specific to rec model training
7. **Optimize** memory usage through strategic precision assignment

## Prerequisites

- Understanding of floating-point number representation
- PyTorch training basics (forward, backward, optimizer)
- Familiarity with recommendation model architectures

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/rec_system/blob/main/notebooks/part9/chapter_9.3_mixed_precision.ipynb)
[![Download](https://img.shields.io/badge/Download-Notebook-blue)](https://github.com/hideak1/rec_system/blob/main/notebooks/part9/chapter_9.3_mixed_precision.ipynb)

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import time
from typing import List, Dict, Tuple, Optional
from collections import defaultdict

np.random.seed(42)
torch.manual_seed(42)

print("All imports successful!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
# We'll simulate GPU behavior on CPU for this notebook

## 1. Floating-Point Formats for Recommendation Training

Understanding the numerical formats is essential for mixed-precision training:

| Format | Bits | Exponent | Mantissa | Range | Precision |
|--------|------|----------|----------|-------|-----------|
| **FP32** | 32 | 8 | 23 | ~1e38 | ~7 decimal digits |
| **FP16** | 16 | 5 | 10 | ~65504 | ~3 decimal digits |
| **BF16** | 16 | 8 | 7 | ~3.4e38 | ~2 decimal digits |
| **INT8** | 8 | - | - | [-128, 127] | - |
| **INT4** | 4 | - | - | [-8, 7] | - |

### Why Mixed-Precision for RecSys?

Recommendation models have unique characteristics:
- **Embedding tables dominate memory**: FP16 embeddings halve memory → fit 2x more items
- **Sparse gradients**: Only a few embeddings update per batch; gradient accumulation needs FP32
- **Interaction features**: Often categorical, can be represented in lower precision

The key equation for mixed-precision training with loss scaling:

$$\text{scaled\_loss} = \text{loss} \times S$$
$$\nabla_{\text{fp16}} = \text{backward}(\text{scaled\_loss})$$
$$\nabla_{\text{fp32}} = \nabla_{\text{fp16}} / S$$

where $S$ is the loss scale factor (typically 2^16 or dynamically adjusted).

> **💡 Concept:** BF16 is often preferred over FP16 for recommendation models because it has the same dynamic range as FP32 (8-bit exponent), avoiding overflow issues common with the narrow range of FP16. Google TPUs natively support BF16.

In [ ]:
# Demonstrate precision differences
def analyze_precision(tensor: torch.Tensor, name: str):
    """Analyze a tensor's precision characteristics."""
    fp32 = tensor.float()
    fp16 = tensor.half()
    bf16 = tensor.bfloat16()
    
    fp16_error = (fp32 - fp16.float()).abs()
    bf16_error = (fp32 - bf16.float()).abs()
    
    print(f"\n{name}:")
    print(f"  FP32 range: [{fp32.min():.6f}, {fp32.max():.6f}]")
    print(f"  FP16 max error: {fp16_error.max():.6f}, mean error: {fp16_error.mean():.6f}")
    print(f"  BF16 max error: {bf16_error.max():.6f}, mean error: {bf16_error.mean():.6f}")
    print(f"  FP16 overflow count: {(fp16.float().abs() == float('inf')).sum().item()}")
    
    return fp16_error, bf16_error


# Typical rec model tensor distributions
embedding_weights = torch.randn(1000, 64) * 0.01  # Small values (Xavier init)
mlp_weights = torch.randn(256, 512) * np.sqrt(2 / 512)  # He init
gradients = torch.randn(1000, 64) * 0.0001  # Very small gradients
large_gradients = torch.randn(100, 64) * 100  # Gradient spikes

results = {}
for tensor, name in [(embedding_weights, 'Embedding Weights'),
                      (mlp_weights, 'MLP Weights'),
                      (gradients, 'Normal Gradients'),
                      (large_gradients, 'Gradient Spikes')]:
    results[name] = analyze_precision(tensor, name)

In [ ]:
# Visualize quantization error distributions
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

tensor_data = [
    (embedding_weights, 'Embedding Weights'),
    (mlp_weights, 'MLP Weights'),
    (gradients, 'Normal Gradients'),
    (large_gradients, 'Gradient Spikes')
]

for idx, (tensor, name) in enumerate(tensor_data):
    ax = axes[idx // 2][idx % 2]
    fp16_err = (tensor.float() - tensor.half().float()).abs().flatten().numpy()
    bf16_err = (tensor.float() - tensor.bfloat16().float()).abs().flatten().numpy()
    
    ax.hist(fp16_err, bins=50, alpha=0.6, label='FP16 error', color='coral', density=True)
    ax.hist(bf16_err, bins=50, alpha=0.6, label='BF16 error', color='steelblue', density=True)
    ax.set_title(f'{name}', fontsize=12)
    ax.set_xlabel('Absolute Error')
    ax.set_ylabel('Density')
    ax.legend()
    ax.set_yscale('log')

plt.suptitle('Quantization Error: FP16 vs BF16', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 2. Mixed-Precision Training for Recommendation Models

We implement mixed-precision training with **dynamic loss scaling** — the standard approach used in production (Micikevicius et al., 2018, NVIDIA).

### The Recipe

1. **Forward pass**: FP16 computation
2. **Loss scaling**: Multiply loss by scale factor $S$
3. **Backward pass**: FP16 gradients (scaled)
4. **Unscale**: Divide gradients by $S$
5. **Check for overflow**: If inf/nan, skip update and reduce $S$
6. **Optimizer step**: FP32 master weights updated with unscaled gradients
7. **Copy**: FP32 master → FP16 model weights

> **⚠️ Common Pitfall:** Embedding gradients in rec models are extremely sparse — most entries are zero. When using FP16, the non-zero gradients can underflow (become zero) because they're often very small. Loss scaling is critical to prevent this.

In [ ]:
class DeepFM(nn.Module):
    """DeepFM model for CTR prediction (Guo et al., 2017, Huawei)."""
    
    def __init__(self, num_features: int, num_fields: int, embedding_dim: int = 16,
                 mlp_dims: List[int] = None):
        super().__init__()
        if mlp_dims is None:
            mlp_dims = [256, 128, 64]
        
        self.embedding = nn.Embedding(num_features, embedding_dim)
        self.linear = nn.Embedding(num_features, 1)
        self.bias = nn.Parameter(torch.zeros(1))
        
        # MLP (deep component)
        input_dim = num_fields * embedding_dim
        layers = []
        for dim in mlp_dims:
            layers.extend([nn.Linear(input_dim, dim), nn.ReLU(), nn.Dropout(0.1)])
            input_dim = dim
        layers.append(nn.Linear(input_dim, 1))
        self.mlp = nn.Sequential(*layers)
        
        self.num_fields = num_fields
        nn.init.xavier_uniform_(self.embedding.weight)
        nn.init.zeros_(self.linear.weight)
    
    def forward(self, feature_ids: torch.Tensor) -> torch.Tensor:
        """feature_ids: (batch_size, num_fields)"""
        # Linear (first-order)
        linear_out = self.linear(feature_ids).sum(dim=1).squeeze() + self.bias
        
        # FM (second-order interactions)
        emb = self.embedding(feature_ids)  # (B, F, D)
        sum_square = emb.sum(dim=1).pow(2)  # (B, D)
        square_sum = emb.pow(2).sum(dim=1)  # (B, D)
        fm_out = 0.5 * (sum_square - square_sum).sum(dim=1)  # (B,)
        
        # Deep
        deep_in = emb.view(emb.size(0), -1)  # (B, F*D)
        deep_out = self.mlp(deep_in).squeeze()  # (B,)
        
        return linear_out + fm_out + deep_out


# Create model and synthetic data
NUM_FEATURES = 100000
NUM_FIELDS = 20
BATCH_SIZE = 512

model = DeepFM(NUM_FEATURES, NUM_FIELDS, embedding_dim=16)
total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")
print(f"FP32 memory: {total_params * 4 / (1024**2):.1f} MB")
print(f"FP16 memory: {total_params * 2 / (1024**2):.1f} MB")

In [ ]:
class MixedPrecisionTrainer:
    """Mixed-precision trainer with dynamic loss scaling."""
    
    def __init__(self, model: nn.Module, lr: float = 0.001,
                 init_scale: float = 2**16, scale_factor: float = 2.0,
                 scale_window: int = 200):
        self.model = model
        # FP32 master weights
        self.master_params = [p.clone().float().detach().requires_grad_(True) 
                              for p in model.parameters()]
        self.optimizer = torch.optim.Adam(self.master_params, lr=lr)
        
        # Dynamic loss scaling
        self.loss_scale = init_scale
        self.scale_factor = scale_factor
        self.scale_window = scale_window
        self.steps_since_last_overflow = 0
        
        # Tracking
        self.overflow_count = 0
        self.scale_history = []
    
    def train_step(self, feature_ids: torch.Tensor, labels: torch.Tensor) -> Dict:
        """Execute one mixed-precision training step."""
        # Copy master (FP32) weights to model (simulate FP16)
        for model_p, master_p in zip(self.model.parameters(), self.master_params):
            model_p.data.copy_(master_p.data)
        
        # Forward pass
        logits = self.model(feature_ids)
        loss = F.binary_cross_entropy_with_logits(logits, labels)
        
        # Scale loss
        scaled_loss = loss * self.loss_scale
        
        # Backward pass
        self.model.zero_grad()
        scaled_loss.backward()
        
        # Check for overflow
        has_overflow = False
        for p in self.model.parameters():
            if p.grad is not None:
                if torch.isnan(p.grad).any() or torch.isinf(p.grad).any():
                    has_overflow = True
                    break
        
        if has_overflow:
            self.overflow_count += 1
            self.loss_scale /= self.scale_factor
            self.steps_since_last_overflow = 0
            self.scale_history.append(self.loss_scale)
            return {'loss': loss.item(), 'overflow': True, 'scale': self.loss_scale}
        
        # Unscale gradients and copy to master
        for model_p, master_p in zip(self.model.parameters(), self.master_params):
            if model_p.grad is not None:
                master_p.grad = model_p.grad.float() / self.loss_scale
        
        # Optimizer step on FP32 master
        self.optimizer.step()
        
        # Update loss scale
        self.steps_since_last_overflow += 1
        if self.steps_since_last_overflow >= self.scale_window:
            self.loss_scale *= self.scale_factor
            self.steps_since_last_overflow = 0
        
        self.scale_history.append(self.loss_scale)
        return {'loss': loss.item(), 'overflow': False, 'scale': self.loss_scale}


# Generate synthetic training data
def generate_ctr_data(n_samples, n_features, n_fields):
    feature_ids = torch.randint(0, n_features, (n_samples, n_fields))
    labels = torch.randint(0, 2, (n_samples,)).float()
    return feature_ids, labels


# Train with mixed precision
model_mp = DeepFM(NUM_FEATURES, NUM_FIELDS, embedding_dim=16)
trainer = MixedPrecisionTrainer(model_mp, lr=0.001)

losses_mp = []
for step in range(200):
    features, labels = generate_ctr_data(BATCH_SIZE, NUM_FEATURES, NUM_FIELDS)
    result = trainer.train_step(features, labels)
    losses_mp.append(result['loss'])
    if step % 50 == 0:
        print(f"Step {step}: loss={result['loss']:.4f}, scale={result['scale']:.0f}, overflow={result['overflow']}")

print(f"\nTotal overflows: {trainer.overflow_count}")
print(f"Final loss scale: {trainer.loss_scale:.0f}")

## 3. Post-Training Quantization for Embeddings

Post-training quantization (PTQ) compresses embedding tables after training, enabling faster inference and smaller model size.

### Quantization Formula

For symmetric quantization to INT8:

$$q(x) = \text{clamp}\left(\text{round}\left(\frac{x}{s}\right), -128, 127\right)$$

where the scale factor $s$ is:

$$s = \frac{\max(|x|)}{127}$$

For dequantization:

$$\hat{x} = q(x) \times s$$

### Per-Channel vs Per-Tensor

- **Per-tensor**: One scale factor for the entire embedding table. Fast but less accurate.
- **Per-channel (per-row)**: One scale factor per embedding row. Better accuracy, standard for embeddings.

> **🔑 Pro Tip:** For embedding tables, per-row quantization is almost always worth the tiny extra cost. Different items can have very different magnitude distributions, and per-row scaling captures this.

In [ ]:
class EmbeddingQuantizer:
    """Post-training quantization for embedding tables."""
    
    @staticmethod
    def quantize_int8_per_tensor(embeddings: torch.Tensor) -> Tuple[torch.Tensor, float]:
        """Symmetric INT8 quantization with per-tensor scale."""
        scale = embeddings.abs().max().item() / 127.0
        quantized = torch.clamp(torch.round(embeddings / scale), -128, 127).to(torch.int8)
        return quantized, scale
    
    @staticmethod
    def dequantize_int8_per_tensor(quantized: torch.Tensor, scale: float) -> torch.Tensor:
        return quantized.float() * scale
    
    @staticmethod
    def quantize_int8_per_row(embeddings: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """Symmetric INT8 quantization with per-row scale."""
        scales = embeddings.abs().max(dim=1).values / 127.0
        scales = torch.clamp(scales, min=1e-8)
        quantized = torch.clamp(
            torch.round(embeddings / scales.unsqueeze(1)), -128, 127
        ).to(torch.int8)
        return quantized, scales
    
    @staticmethod
    def dequantize_int8_per_row(quantized: torch.Tensor, scales: torch.Tensor) -> torch.Tensor:
        return quantized.float() * scales.unsqueeze(1)
    
    @staticmethod
    def quantize_int4_per_row(embeddings: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """Symmetric INT4 quantization (simulated with INT8 storage)."""
        scales = embeddings.abs().max(dim=1).values / 7.0
        scales = torch.clamp(scales, min=1e-8)
        quantized = torch.clamp(
            torch.round(embeddings / scales.unsqueeze(1)), -8, 7
        ).to(torch.int8)  # INT4 packed in INT8 for simulation
        return quantized, scales
    
    @staticmethod
    def dequantize_int4_per_row(quantized: torch.Tensor, scales: torch.Tensor) -> torch.Tensor:
        return quantized.float() * scales.unsqueeze(1)


# Create a trained embedding table (simulate post-training state)
torch.manual_seed(42)
num_items = 50000
emb_dim = 64
trained_embeddings = torch.randn(num_items, emb_dim) * 0.1

# Make some rows have different scales (realistic)
for i in range(0, num_items, 1000):
    trained_embeddings[i:i+100] *= 10.0  # Popular items may have larger embeddings

quantizer = EmbeddingQuantizer()

# Quantize with different methods
q_pt, s_pt = quantizer.quantize_int8_per_tensor(trained_embeddings)
q_pr, s_pr = quantizer.quantize_int8_per_row(trained_embeddings)
q_i4, s_i4 = quantizer.quantize_int4_per_row(trained_embeddings)

# Dequantize and compute errors
dq_pt = quantizer.dequantize_int8_per_tensor(q_pt, s_pt)
dq_pr = quantizer.dequantize_int8_per_row(q_pr, s_pr)
dq_i4 = quantizer.dequantize_int4_per_row(q_i4, s_i4)

err_pt = (trained_embeddings - dq_pt).abs()
err_pr = (trained_embeddings - dq_pr).abs()
err_i4 = (trained_embeddings - dq_i4).abs()

print(f"{'Method':<25} {'Mean Error':<15} {'Max Error':<15} {'Memory (MB)':<15}")
print("-" * 70)
print(f"{'FP32 (baseline)':<25} {'0':<15} {'0':<15} {num_items * emb_dim * 4 / (1024**2):<15.1f}")
print(f"{'INT8 per-tensor':<25} {err_pt.mean():.6f}{'':>8} {err_pt.max():.6f}{'':>8} {(q_pt.numel() + 4) / (1024**2):<15.1f}")
print(f"{'INT8 per-row':<25} {err_pr.mean():.6f}{'':>8} {err_pr.max():.6f}{'':>8} {(q_pr.numel() + s_pr.numel() * 4) / (1024**2):<15.1f}")
print(f"{'INT4 per-row':<25} {err_i4.mean():.6f}{'':>8} {err_i4.max():.6f}{'':>8} {(q_i4.numel() * 0.5 + s_i4.numel() * 4) / (1024**2):<15.1f}")

In [ ]:
# Visualize quantization error distributions
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Per-row error comparison
row_errors_pt = err_pt.mean(dim=1).numpy()
row_errors_pr = err_pr.mean(dim=1).numpy()
row_errors_i4 = err_i4.mean(dim=1).numpy()

axes[0].hist(row_errors_pt, bins=50, alpha=0.6, label='INT8 per-tensor', color='coral')
axes[0].hist(row_errors_pr, bins=50, alpha=0.6, label='INT8 per-row', color='steelblue')
axes[0].hist(row_errors_i4, bins=50, alpha=0.6, label='INT4 per-row', color='green')
axes[0].set_title('Per-Row Mean Absolute Error', fontsize=12)
axes[0].set_xlabel('Mean Absolute Error')
axes[0].set_ylabel('Count')
axes[0].legend()

# Cosine similarity preservation
n_pairs = 1000
idx_a = torch.randint(0, num_items, (n_pairs,))
idx_b = torch.randint(0, num_items, (n_pairs,))

cos_orig = F.cosine_similarity(trained_embeddings[idx_a], trained_embeddings[idx_b], dim=1)
cos_int8 = F.cosine_similarity(dq_pr[idx_a], dq_pr[idx_b], dim=1)
cos_int4 = F.cosine_similarity(dq_i4[idx_a], dq_i4[idx_b], dim=1)

axes[1].scatter(cos_orig.numpy(), cos_int8.numpy(), alpha=0.3, s=5, label='INT8', color='steelblue')
axes[1].scatter(cos_orig.numpy(), cos_int4.numpy(), alpha=0.3, s=5, label='INT4', color='green')
axes[1].plot([-1, 1], [-1, 1], 'r--', alpha=0.5)
axes[1].set_xlabel('Original Cosine Similarity')
axes[1].set_ylabel('Quantized Cosine Similarity')
axes[1].set_title('Similarity Preservation', fontsize=12)
axes[1].legend()

# Memory comparison
methods = ['FP32', 'FP16', 'INT8\nper-row', 'INT4\nper-row']
memory_mb = [
    num_items * emb_dim * 4 / (1024**2),
    num_items * emb_dim * 2 / (1024**2),
    (num_items * emb_dim + num_items * 4) / (1024**2),
    (num_items * emb_dim * 0.5 + num_items * 4) / (1024**2),
]
colors = ['gray', 'steelblue', 'coral', 'green']
axes[2].bar(methods, memory_mb, color=colors, alpha=0.8)
axes[2].set_ylabel('Memory (MB)')
axes[2].set_title('Memory Footprint by Precision', fontsize=12)
for i, v in enumerate(memory_mb):
    axes[2].text(i, v + 0.3, f'{v:.1f}', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

## 4. Quantization-Aware Training (QAT)

QAT simulates quantization during training, allowing the model to learn to be robust to quantization noise.

### Straight-Through Estimator (STE)

The quantize operation is non-differentiable (rounding). QAT uses the **Straight-Through Estimator** (Bengio et al., 2013):

**Forward**: $\hat{x} = \text{dequantize}(\text{quantize}(x))$ (with quantization noise)

**Backward**: $\frac{\partial \hat{x}}{\partial x} = 1$ (pass gradient through as if identity)

> **💡 Concept:** QAT typically recovers 50-90% of the accuracy gap between full-precision and post-training quantization. For embeddings, this means INT8 QAT can be nearly lossless while INT4 QAT can match INT8 PTQ quality.

In [ ]:
class QuantizedEmbedding(nn.Module):
    """Embedding with simulated quantization (QAT) using straight-through estimator."""
    
    def __init__(self, num_embeddings: int, embedding_dim: int, bits: int = 8):
        super().__init__()
        self.embedding = nn.Embedding(num_embeddings, embedding_dim)
        nn.init.xavier_uniform_(self.embedding.weight)
        self.bits = bits
        self.qmin = -(2 ** (bits - 1))
        self.qmax = 2 ** (bits - 1) - 1
    
    def _fake_quantize(self, x: torch.Tensor) -> torch.Tensor:
        """Simulate quantization with STE."""
        # Per-row scale
        scale = x.abs().max(dim=-1, keepdim=True).values / self.qmax
        scale = torch.clamp(scale, min=1e-8)
        
        # Quantize
        x_q = torch.clamp(torch.round(x / scale), self.qmin, self.qmax)
        
        # Dequantize
        x_dq = x_q * scale
        
        # STE: forward uses quantized, backward passes through
        return x + (x_dq - x).detach()
    
    def forward(self, indices: torch.Tensor) -> torch.Tensor:
        emb = self.embedding(indices)
        if self.training:
            emb = self._fake_quantize(emb)
        return emb


class DeepFM_QAT(nn.Module):
    """DeepFM with quantization-aware training for embeddings."""
    
    def __init__(self, num_features: int, num_fields: int, embedding_dim: int = 16,
                 bits: int = 8):
        super().__init__()
        self.embedding = QuantizedEmbedding(num_features, embedding_dim, bits=bits)
        self.linear = nn.Embedding(num_features, 1)
        self.bias = nn.Parameter(torch.zeros(1))
        self.num_fields = num_fields
        
        input_dim = num_fields * embedding_dim
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, 128), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(64, 1)
        )
    
    def forward(self, feature_ids: torch.Tensor) -> torch.Tensor:
        linear_out = self.linear(feature_ids).sum(dim=1).squeeze() + self.bias
        emb = self.embedding(feature_ids)
        sum_sq = emb.sum(dim=1).pow(2)
        sq_sum = emb.pow(2).sum(dim=1)
        fm_out = 0.5 * (sum_sq - sq_sum).sum(dim=1)
        deep_out = self.mlp(emb.view(emb.size(0), -1)).squeeze()
        return linear_out + fm_out + deep_out


# Compare FP32, QAT-INT8, QAT-INT4
def train_model(model, n_steps=200, batch_size=512, n_features=10000, n_fields=20):
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    losses = []
    for step in range(n_steps):
        features = torch.randint(0, n_features, (batch_size, n_fields))
        labels = torch.randint(0, 2, (batch_size,)).float()
        logits = model(features)
        loss = F.binary_cross_entropy_with_logits(logits, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    return losses


torch.manual_seed(42)
model_fp32 = DeepFM(10000, 20, embedding_dim=16, mlp_dims=[128, 64])
torch.manual_seed(42)
model_qat8 = DeepFM_QAT(10000, 20, embedding_dim=16, bits=8)
torch.manual_seed(42)
model_qat4 = DeepFM_QAT(10000, 20, embedding_dim=16, bits=4)

print("Training FP32...")
losses_fp32 = train_model(model_fp32)
print("Training QAT-INT8...")
losses_qat8 = train_model(model_qat8)
print("Training QAT-INT4...")
losses_qat4 = train_model(model_qat4)

print(f"\nFinal losses:")
print(f"  FP32:     {np.mean(losses_fp32[-20:]):.4f}")
print(f"  QAT-INT8: {np.mean(losses_qat8[-20:]):.4f}")
print(f"  QAT-INT4: {np.mean(losses_qat4[-20:]):.4f}")

In [ ]:
# Training curve comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

window = 10
smooth = lambda x: np.convolve(x, np.ones(window)/window, mode='valid')

axes[0].plot(smooth(losses_fp32), label='FP32', color='gray', linewidth=2)
axes[0].plot(smooth(losses_qat8), label='QAT-INT8', color='steelblue', linewidth=2)
axes[0].plot(smooth(losses_qat4), label='QAT-INT4', color='coral', linewidth=2)
axes[0].set_xlabel('Training Step', fontsize=12)
axes[0].set_ylabel('Loss (smoothed)', fontsize=12)
axes[0].set_title('Training Convergence: FP32 vs QAT', fontsize=13)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Loss scale history from mixed-precision training
axes[1].plot(trainer.scale_history, color='steelblue', linewidth=1.5)
axes[1].set_xlabel('Training Step', fontsize=12)
axes[1].set_ylabel('Loss Scale', fontsize=12)
axes[1].set_title('Dynamic Loss Scale History', fontsize=13)
axes[1].set_yscale('log')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Mixed-Precision Embedding Lookups with FP32 Accumulation

A practical technique used at Meta and Google: store embeddings in FP16/INT8 but **accumulate in FP32** during the forward pass.

This is critical for features like `EmbeddingBag` (sum/mean of multiple embeddings) where accumulation errors compound:

$$\text{EmbeddingBag}(\{i_1, ..., i_k\}) = \sum_{j=1}^{k} e_{i_j}$$

With FP16 accumulation, the error grows as $O(\sqrt{k})$ where $k$ is the bag size. For features with large bag sizes (e.g., user history with 1000+ items), this becomes significant.

> **⚠️ Common Pitfall:** TorchRec's `EmbeddingBagCollection` defaults to FP32 accumulation even when embeddings are stored in FP16. If you implement your own, do not forget this critical detail.

In [ ]:
class MixedPrecisionEmbeddingBag:
    """Embedding bag with FP16 storage and FP32 accumulation."""
    
    def __init__(self, num_embeddings: int, embedding_dim: int, mode: str = 'sum'):
        # Store in FP16
        self.weight_fp16 = (torch.randn(num_embeddings, embedding_dim) * 0.01).half()
        self.mode = mode
    
    def forward_fp16_accum(self, indices: torch.Tensor, offsets: torch.Tensor) -> torch.Tensor:
        """Accumulate in FP16 (potentially lossy)."""
        results = []
        for i in range(len(offsets) - 1):
            bag_indices = indices[offsets[i]:offsets[i+1]]
            embs = self.weight_fp16[bag_indices]
            if self.mode == 'sum':
                results.append(embs.sum(dim=0))  # FP16 accumulation
            else:
                results.append(embs.mean(dim=0))
        return torch.stack(results)
    
    def forward_fp32_accum(self, indices: torch.Tensor, offsets: torch.Tensor) -> torch.Tensor:
        """Accumulate in FP32 (recommended)."""
        results = []
        for i in range(len(offsets) - 1):
            bag_indices = indices[offsets[i]:offsets[i+1]]
            embs = self.weight_fp16[bag_indices].float()  # Cast to FP32 before accumulation
            if self.mode == 'sum':
                results.append(embs.sum(dim=0))
            else:
                results.append(embs.mean(dim=0))
        return torch.stack(results).half()  # Cast back to FP16


# Demonstrate the difference with increasing bag sizes
emb_bag = MixedPrecisionEmbeddingBag(10000, 64)

# Ground truth: FP32 everywhere
weight_fp32 = emb_bag.weight_fp16.float()

bag_sizes = [1, 5, 10, 50, 100, 500, 1000]
fp16_errors = []
fp32_errors = []

for bag_size in bag_sizes:
    n_bags = 100
    indices = torch.randint(0, 10000, (n_bags * bag_size,))
    offsets = torch.arange(0, n_bags * bag_size + 1, bag_size)
    
    # Ground truth
    gt_results = []
    for i in range(n_bags):
        bag_idx = indices[offsets[i]:offsets[i+1]]
        gt_results.append(weight_fp32[bag_idx].sum(dim=0))
    gt = torch.stack(gt_results)
    
    # FP16 accumulation
    fp16_result = emb_bag.forward_fp16_accum(indices, offsets).float()
    fp16_err = (gt - fp16_result).abs().mean().item()
    fp16_errors.append(fp16_err)
    
    # FP32 accumulation
    fp32_result = emb_bag.forward_fp32_accum(indices, offsets).float()
    fp32_err = (gt - fp32_result).abs().mean().item()
    fp32_errors.append(fp32_err)

print(f"{'Bag Size':<12} {'FP16 Accum Error':<20} {'FP32 Accum Error':<20} {'Ratio':<10}")
print("-" * 60)
for bs, e16, e32 in zip(bag_sizes, fp16_errors, fp32_errors):
    ratio = e16 / max(e32, 1e-10)
    print(f"{bs:<12} {e16:<20.6f} {e32:<20.6f} {ratio:<10.1f}x")

## 6. Benchmarking: Speed vs Accuracy Trade-offs

Let us run a comprehensive benchmark comparing different precision configurations on a DeepFM model.

In [ ]:
def benchmark_precision(model, n_steps=100, batch_size=512, n_features=10000, n_fields=20):
    """Benchmark training speed and final loss for a given model."""
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    
    # Warmup
    for _ in range(10):
        features = torch.randint(0, n_features, (batch_size, n_fields))
        labels = torch.randint(0, 2, (batch_size,)).float()
        logits = model(features)
        loss = F.binary_cross_entropy_with_logits(logits, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    # Timed run
    losses = []
    t0 = time.time()
    for _ in range(n_steps):
        features = torch.randint(0, n_features, (batch_size, n_fields))
        labels = torch.randint(0, 2, (batch_size,)).float()
        logits = model(features)
        loss = F.binary_cross_entropy_with_logits(logits, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    elapsed = time.time() - t0
    
    return {
        'time_per_step_ms': elapsed / n_steps * 1000,
        'throughput_samples_per_sec': n_steps * batch_size / elapsed,
        'final_loss': np.mean(losses[-20:]),
        'memory_mb': sum(p.numel() * p.element_size() for p in model.parameters()) / (1024**2)
    }


configs = {
    'FP32 (baseline)': DeepFM(10000, 20, embedding_dim=16, mlp_dims=[128, 64]),
    'QAT-INT8': DeepFM_QAT(10000, 20, embedding_dim=16, bits=8),
    'QAT-INT4': DeepFM_QAT(10000, 20, embedding_dim=16, bits=4),
}

results = {}
for name, model in configs.items():
    torch.manual_seed(42)
    results[name] = benchmark_precision(model)
    print(f"{name}: {results[name]}")

# Summary table
print(f"\n{'Config':<20} {'Time/Step (ms)':<17} {'Throughput':<15} {'Loss':<10} {'Memory (MB)':<12}")
print("-" * 75)
for name, r in results.items():
    print(f"{name:<20} {r['time_per_step_ms']:<17.2f} {r['throughput_samples_per_sec']:<15.0f} {r['final_loss']:<10.4f} {r['memory_mb']:<12.2f}")

## 🏋️ Exercise 1: Implement Mixed-Precision Training for DeepFM

Implement a complete mixed-precision training loop with dynamic loss scaling for the DeepFM model. Track gradient norms at each step to detect potential instabilities.

In [ ]:
# 🏋️ Exercise 1: Mixed-precision DeepFM training

def train_deepfm_mixed_precision(
    num_features: int = 10000,
    num_fields: int = 20,
    embedding_dim: int = 16,
    batch_size: int = 512,
    n_steps: int = 300,
    lr: float = 0.001,
    init_scale: float = 2**15
) -> Dict:
    """
    Train DeepFM with mixed precision.
    
    Returns:
        Dict with 'losses', 'grad_norms', 'scale_history', 'overflow_count'
    """
    # TODO:
    # 1. Create DeepFM model
    # 2. Create FP32 master weights copy
    # 3. Training loop with:
    #    a. Forward in FP16 (simulated)
    #    b. Loss scaling
    #    c. Backward pass
    #    d. Overflow detection
    #    e. Gradient unscaling
    #    f. FP32 optimizer step
    #    g. Dynamic scale adjustment
    # 4. Track gradient norms at each step
    pass


# Test:
# result = train_deepfm_mixed_precision()
# print(f"Final loss: {np.mean(result['losses'][-20:]):.4f}")
# print(f"Overflow count: {result['overflow_count']}")

## 🏋️ Exercise 2: Implement Adaptive Quantization for Embeddings

Implement an adaptive quantization scheme that uses different bit-widths for different embedding rows based on their importance (access frequency).

In [ ]:
# 🏋️ Exercise 2: Adaptive quantization

class AdaptiveQuantizedEmbedding:
    def __init__(self, embeddings: torch.Tensor, access_counts: np.ndarray,
                 high_bits: int = 8, low_bits: int = 4, threshold_percentile: float = 80):
        """
        Adaptive quantization: frequent items get higher precision.
        
        Args:
            embeddings: (N, D) FP32 embedding weights
            access_counts: (N,) access frequency for each embedding
            high_bits: Bit-width for frequent items
            low_bits: Bit-width for infrequent items
            threshold_percentile: Percentile threshold for high/low split
        """
        # TODO:
        # 1. Determine which rows are "high frequency" based on threshold
        # 2. Quantize high-freq rows with high_bits
        # 3. Quantize low-freq rows with low_bits
        # 4. Store scales and quantized values
        pass
    
    def lookup(self, indices: torch.Tensor) -> torch.Tensor:
        """
        Look up and dequantize embeddings.
        
        Returns:
            (len(indices), D) FP32 tensor
        """
        # TODO: Dequantize with appropriate scales based on index tier
        pass
    
    def memory_savings(self) -> float:
        """Return memory savings ratio vs FP32."""
        # TODO: Calculate actual memory used vs FP32 baseline
        pass


# Test:
# embs = torch.randn(10000, 64) * 0.1
# counts = np.random.zipf(1.5, 10000).astype(float)
# aq = AdaptiveQuantizedEmbedding(embs, counts)
# result = aq.lookup(torch.tensor([0, 100, 5000]))
# print(f"Result shape: {result.shape}")
# print(f"Memory savings: {aq.memory_savings():.1%}")

## 🏋️ Exercise 3: Benchmark Quantization Impact on Ranking Quality

Train a recommendation model, then quantize it with different methods and measure the impact on ranking metrics (NDCG, recall).

In [ ]:
# 🏋️ Exercise 3: Quantization impact on ranking

def evaluate_quantization_ranking_impact(
    num_users: int = 1000,
    num_items: int = 5000,
    embedding_dim: int = 32,
    n_train_steps: int = 200
) -> Dict:
    """
    Train a two-tower model, quantize embeddings, and compare ranking quality.
    
    Returns:
        Dict with ranking metrics for each quantization method
    """
    # TODO:
    # 1. Create a two-tower model (user embedding + item embedding + dot product)
    # 2. Generate synthetic interaction data
    # 3. Train the model
    # 4. Evaluate ranking quality (NDCG@10, Recall@10) with:
    #    a. FP32 (baseline)
    #    b. INT8 per-row quantization
    #    c. INT4 per-row quantization
    # 5. Return comparison metrics
    pass


# Test:
# metrics = evaluate_quantization_ranking_impact()
# for method, m in metrics.items():
#     print(f"{method}: NDCG@10={m['ndcg']:.4f}, Recall@10={m['recall']:.4f}")

## Summary

| Topic | Key Takeaway |
|-------|--------------|
| **FP16 vs BF16** | BF16 preferred for rec models due to wider dynamic range (avoids overflow) |
| **Dynamic Loss Scaling** | Essential for FP16; auto-adjusts to prevent underflow/overflow |
| **Post-Training Quantization** | INT8 per-row is nearly lossless; INT4 has measurable but often acceptable quality loss |
| **QAT** | Recovers 50-90% of PTQ quality gap; recommended for INT4 |
| **FP32 Accumulation** | Critical for embedding bags with large bag sizes |
| **Memory Savings** | INT8 = 4x, INT4 = 8x reduction in embedding table memory |

### Key References
- Micikevicius et al., "Mixed Precision Training" (2018, NVIDIA/Baidu)
- Guo et al., "DeepFM: A Factorization-Machine based Neural Network for CTR Prediction" (2017, Huawei)
- Jacob et al., "Quantization and Training of Neural Networks for Efficient Integer-Arithmetic-Only Inference" (2018, Google)
- Dettmers et al., "LLM.int8(): 8-bit Matrix Multiplication for Transformers at Scale" (2022)

### Next Steps
In the next notebook (9.4), we will explore embedding training optimization techniques including lazy updates, frequency-based learning rates, and adaptive dimensions.